In [1]:
import pandas as pd
from tqdm.auto import tqdm
import sys
sys.path.append("../")
tqdm.pandas()

In [2]:
from src.repo_utils import clone_and_extract_tree

In [3]:
CONTEXT_WINDOW = 395000 # gpt 5.2's context window is 400k tokens

In [4]:
df_repos = pd.read_parquet("../data/02_paper_assessment/references_paper_assessment_pred.parquet.br")

In [7]:
# subset rows where LLM_url is present and not the literal "Appendix" (case-insensitive)
df_repos_assessment = df_repos[
    df_repos["url"].notna() &
    (df_repos["url"].astype(str).str.strip().str.lower() != "appendix")
].copy()

In [ ]:
def wrapper_clone_and_extract_tree(repo_url):
    result = clone_and_extract_tree(
        repo_url=repo_url,
        context_window=CONTEXT_WINDOW,
        clone_dir="../data/03_repo_assessment/cloned_repos",
    )
    return {
        "repo_content": result.output,
        "repo_status": result.status,
    }

In [ ]:
df_repos_assessment[["repo_content", "repo_status"]] = (
    df_repos_assessment["url"]
    .progress_apply(wrapper_clone_and_extract_tree)
    .apply(pd.Series)
)